# 2.empirical_analysis_15test.py

Autogenerated from the matching source file in `14.15Test/code_src`.

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from scipy import stats
from sklearn.metrics import auc, roc_curve

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

ROOT = Path("/Users/jane/Documents/202511吾-Systems/14.15Test")
MERGED_DIR = ROOT / "merged"
TABLE_DIR = ROOT / "tables"
PLOT_DIR = ROOT / "plots"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

file_path = MERGED_DIR / "TDA_results_merged_21_1.csv"
events = pd.read_csv(MERGED_DIR / "crypto_market_events.csv", parse_dates=["event_date"])["event_date"]
df = pd.read_csv(file_path, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True).set_index("date")

window = 21
tda_indicators = [
    "vol_wass_betti0", "vol_wass_betti1", "vol_wass_entropy",
    "log_wass_betti0", "log_wass_betti1", "log_wass_entropy",
    "log_dtw_betti0", "log_dtw_betti1", "log_dtw_entropy",
    "vol_dtw_betti0", "vol_dtw_betti1", "vol_dtw_entropy",
]

color_map = {
    "vol_wass": "#E64B35FF",
    "log_wass": "#4DBBD5FF",
    "vol_dtw": "#00A087FF",
    "log_dtw": "#3C5488FF",
}
feature_ls_map = {"betti0": "-", "betti1": "--", "entropy": ":"}

before_after_rows = []
pre_baseline_rows = []

for ind in tda_indicators:
    event_rows_1 = []
    event_rows_2 = []
    for e in events:
        if e not in df.index:
            continue
        before = df.loc[e - pd.Timedelta(days=window): e, ind]
        after = df.loc[e: e + pd.Timedelta(days=window), ind]
        baseline = df.loc[e - pd.Timedelta(days=2 * window): e - pd.Timedelta(days=window), ind]
        pre = df.loc[e - pd.Timedelta(days=window): e, ind]

        if len(before) >= 5 and len(after) >= 5:
            event_rows_1.append({
                "indicator": ind,
                "event": e,
                "before_mean": before.mean(),
                "after_mean": after.mean(),
                "change_after_minus_before": after.mean() - before.mean(),
                "before_std": before.std(),
                "after_std": after.std(),
                "n_before": len(before),
                "n_after": len(after),
            })

        if len(pre) >= 10 and len(baseline) >= 10 and len(pre) == len(baseline):
            t_stat, p_val = stats.ttest_rel(pre, baseline, alternative="greater")
            event_rows_2.append({
                "indicator": ind,
                "event": e,
                "baseline_mean": baseline.mean(),
                "pre_mean": pre.mean(),
                "difference": pre.mean() - baseline.mean(),
                "t_stat": t_stat,
                "p_value": p_val,
            })

    df1 = pd.DataFrame(event_rows_1)
    if not df1.empty:
        t_stat, p_val = stats.ttest_1samp(df1["change_after_minus_before"], popmean=0)
        before_after_rows.append({
            "indicator": ind,
            "mean_change_after_minus_before": df1["change_after_minus_before"].mean(),
            "hit_rate_change_gt_0": (df1["change_after_minus_before"] > 0).mean(),
            "t_stat": t_stat,
            "p_value": p_val,
            "n_events": len(df1),
        })
        df1.to_csv(TABLE_DIR / f"empirical_before_after_events_{ind}.csv", index=False)

    df2 = pd.DataFrame(event_rows_2)
    if not df2.empty:
        pre_baseline_rows.append({
            "indicator": ind,
            "mean_difference_pre_minus_baseline": df2["difference"].mean(),
            "significant_positive_rate": (df2["p_value"] < 0.05).mean(),
            "mean_p_value": df2["p_value"].mean(),
            "n_events": len(df2),
        })
        df2.to_csv(TABLE_DIR / f"empirical_pre_vs_baseline_events_{ind}.csv", index=False)

before_after_summary = pd.DataFrame(before_after_rows).sort_values(
    "mean_change_after_minus_before", ascending=False
)
pre_baseline_summary = pd.DataFrame(pre_baseline_rows).sort_values(
    "significant_positive_rate", ascending=False
)
before_after_summary.to_csv(TABLE_DIR / "empirical_before_after_summary_15.csv", index=False)
pre_baseline_summary.to_csv(TABLE_DIR / "empirical_pre_vs_baseline_summary_15.csv", index=False)

roc_df = df.copy()
roc_df["event_label"] = 0
for e in events:
    roc_df.loc[e - pd.Timedelta(days=window): e + pd.Timedelta(days=window), "event_label"] = 1

auc_results = []
y_true = roc_df["event_label"].astype(int).to_numpy()
for ind in tda_indicators:
    y_score = roc_df[ind].to_numpy(dtype=float)
    mask = np.isfinite(y_score)
    fpr, tpr, _ = roc_curve(y_true[mask], y_score[mask])
    roc_auc = auc(fpr, tpr)
    auc_results.append({"indicator": ind, "auc": roc_auc})

auc_df = pd.DataFrame(auc_results).sort_values("auc", ascending=False)
auc_df.to_csv(TABLE_DIR / "auc_tda_results_15.csv", index=False)


def get_color(name):
    if "vol_wass" in name:
        return color_map["vol_wass"]
    if "log_wass" in name:
        return color_map["log_wass"]
    if "vol_dtw" in name:
        return color_map["vol_dtw"]
    if "log_dtw" in name:
        return color_map["log_dtw"]
    return "#999999"


sorted_items = list(auc_df[["indicator", "auc"]].itertuples(index=False, name=None))
names = [x[0] for x in sorted_items]
auc_vals = [x[1] for x in sorted_items]
colors = [get_color(n) for n in names]
labels = [n.replace("_", " ") for n in names]

fig, ax = plt.subplots(figsize=(14, 8))
x_pos = np.arange(len(names))
bars = ax.bar(x_pos, auc_vals, color=colors, edgecolor="black", linewidth=0.7, alpha=0.9)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=12, rotation=45, ha="right")
ax.set_ylim(max(0.45, min(auc_vals) - 0.03), min(0.9, max(auc_vals) + 0.04))
for bar, value in zip(bars, auc_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=11,
    )
ax.set_ylabel("AUC", fontsize=14)
ax.set_title("AUC of topological indicators (15 assets)", fontsize=18)
ax.yaxis.grid(True, linestyle="--", alpha=0.3)
legend_handles = [
    Patch(color=color_map["vol_wass"], label="Vol–Wasserstein"),
    Patch(color=color_map["log_wass"], label="Log–Wasserstein"),
    Patch(color=color_map["vol_dtw"], label="Vol–DTW"),
    Patch(color=color_map["log_dtw"], label="Log–DTW"),
]
ax.legend(handles=legend_handles, fontsize=14, frameon=False, loc="upper right")
plt.tight_layout()
fig.savefig(PLOT_DIR / "fig_auc_tda_15.png", dpi=900, bbox_inches="tight")
fig.savefig(PLOT_DIR / "fig_auc_tda_15.pdf", dpi=900, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 8))
for col in tda_indicators:
    if "vol_wass" in col:
        color = color_map["vol_wass"]
    elif "log_wass" in col:
        color = color_map["log_wass"]
    elif "vol_dtw" in col:
        color = color_map["vol_dtw"]
    else:
        color = color_map["log_dtw"]

    if "betti0" in col:
        ls = feature_ls_map["betti0"]
    elif "betti1" in col:
        ls = feature_ls_map["betti1"]
    else:
        ls = feature_ls_map["entropy"]

    y_score = roc_df[col].to_numpy(dtype=float)
    mask = np.isfinite(y_score)
    fpr, tpr, _ = roc_curve(y_true[mask], y_score[mask])
    ax.plot(fpr, tpr, label=col.replace("_", " "), color=color, linestyle=ls, linewidth=2)

ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1.2)
ax.set_xlabel("False positive rate", fontsize=13)
ax.set_ylabel("True positive rate", fontsize=13)
ax.set_title("ROC curves for topological indicators (15 assets)", fontsize=15)
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(
    loc="lower right",
    fontsize=10,
    frameon=True,
    facecolor="white",
    framealpha=0.85,
    edgecolor="lightgray",
)
plt.tight_layout()
fig.savefig(PLOT_DIR / "fig_roc_tda_15.png", dpi=900, bbox_inches="tight")
fig.savefig(PLOT_DIR / "fig_roc_tda_15.pdf", dpi=900, bbox_inches="tight")
plt.close(fig)

manifest = pd.DataFrame([
    {"file": "empirical_before_after_summary_15.csv", "path": str(TABLE_DIR / "empirical_before_after_summary_15.csv")},
    {"file": "empirical_pre_vs_baseline_summary_15.csv", "path": str(TABLE_DIR / "empirical_pre_vs_baseline_summary_15.csv")},
    {"file": "auc_tda_results_15.csv", "path": str(TABLE_DIR / "auc_tda_results_15.csv")},
    {"file": "fig_auc_tda_15.png", "path": str(PLOT_DIR / "fig_auc_tda_15.png")},
    {"file": "fig_roc_tda_15.png", "path": str(PLOT_DIR / "fig_roc_tda_15.png")},
])
manifest.to_csv(TABLE_DIR / "empirical_analysis_manifest_15.csv", index=False)

print("Empirical analysis finished.")
print(auc_df.head())


Empirical analysis finished.
         indicator       auc
9   vol_dtw_betti0  0.664445
3  log_wass_betti0  0.661094
6   log_dtw_betti0  0.657995
0  vol_wass_betti0  0.652975
7   log_dtw_betti1  0.546561
